# 🎯 Objetivos

El objetivo de este script es construir y evaluar un modelo de recomendación basado en contenido usando la información textual y estructural de los videojuegos. 

* ⚙️ **Configuración inicial**. Rutas, paquetes y funciones.
* ♻️ **Carga de datos**.
* ✅ **Chequear la integridad**.
* 🧊 **Perfil de usuario**.
* 📈 **Evaluación del modelo**.
* 💾 **Guardado de información**.

# ⚙️ Configuración

## Rutas (paths)

In [1]:
import os

# Rutas en función de la carpeta raíz del proyecto (README.md)
base_path = os.path.dirname(os.path.dirname(os.path.abspath(__file__))) if "__file__" in locals() else os.path.abspath("..")

# Subrutas
data_dir = os.path.join(base_path, "data")
db_path = os.path.join(data_dir, "steam.db")
src_dir = os.path.join(base_path, "src")
models_dir = os.path.join(base_path, "models")
models_content_dir = os.path.join(models_dir, "content")
results_dir = os.path.join(base_path, "res")

## Paquetes y funciones

In [ ]:
# Sistema y configuración
import warnings
warnings.filterwarnings("ignore")
import sqlite3

# Análisis y manipulación de datos
import pandas as pd
import numpy as np

# Matrices
from scipy import sparse
from scipy.sparse import hstack

# Modelos de machine learning
from sklearn.preprocessing import normalize

# Funciones personalizadas    
from utils.evaluation_functions import evaluate_content_user_LMO_at_k

# ♻️ Carga de datos

In [3]:
# Base escalada (con todas las variables)
conn = sqlite3.connect(db_path)
cur = conn.cursor()

games = pd.read_sql("SELECT * FROM games_escaled_final", sqlite3.connect(db_path))

conn.close()

# Hacemos una copia para trabajar
games = games.copy()

In [4]:
# Base escalada user_games_final
conn = sqlite3.connect(db_path)
cur = conn.cursor()

user_games = pd.read_sql("SELECT * FROM user_games_final", sqlite3.connect(db_path))

conn.close()

# Hacemos una copia para trabajar
user_games = user_games.copy()

In [5]:
# Tablas test - train
conn = sqlite3.connect(db_path)
cur = conn.cursor()

user_train = pd.read_sql("SELECT * FROM user_train_LMO", sqlite3.connect(db_path))
user_test = pd.read_sql("SELECT * FROM user_test_LMO", sqlite3.connect(db_path))

conn.close()

# Hacemos una copia para trabajar
user_train = user_train.copy()
user_test = user_test.copy()

In [6]:
# Normalización de tipos
user_games["user_id"] = user_games["user_id"].astype(np.int64)
user_games["appid"] = user_games["appid"].astype(np.int64)
user_train["user_id"] = user_train["user_id"].astype(np.int64)
user_train["appid"] = user_train["appid"].astype(np.int64)
user_test["user_id"] = user_test["user_id"].astype(np.int64)
user_test["appid"] = user_test["appid"].astype(np.int64)
games["appid"] = games["appid"].astype(np.int64)

In [7]:
# Matriz TF-IDF dispersa --> Variables textuales
X_text  = sparse.load_npz(os.path.join(models_dir, "tfidf_matrix.npz"))

# Matriz estructural normalizada dispersa (para combinar con TF-IDF) --> Variables estructurales
X_struct = sparse.load_npz(os.path.join(models_dir, "X_struct.npz"))

# Índice de juegos
idx = pd.read_csv(os.path.join(models_dir, "games_index.csv")) 
idx["appid"] = idx["appid"].astype(np.int64)


In [ ]:
# Mantener solo juegos presentes en idx
valid_appids = set(idx["appid"])

user_train = user_train[user_train["appid"].isin(valid_appids)].copy()
user_test  = user_test[user_test["appid"].isin(valid_appids)].copy()

print("Usuarios tras filtrado:", len(set(user_train["user_id"])))
print("Interacciones train:", len(user_train))
print("Interacciones test:", len(user_test))

Usuarios tras filtrado: 47627
Interacciones train: 6847670
Interacciones test: 142881


# ✅ Chequeos

En este apartado verificamos que las fuentes de datos y los artefactos cargados están correctamente alineados y son consistentes entre sí antes de construir el modelo de contenido.

In [9]:
# Duplicados
assert games["appid"].is_unique, "Hay appid duplicados en games"
assert idx["appid"].is_unique,   "Hay appid duplicados en idx"

In [10]:
# Verificar forma de las matrices y el índice
print("Formas:")
print("X_text  :", X_text.shape)
print("X_struct:", X_struct.shape)
print("idx     :", idx.shape)
print("games   :", games.shape)

assert X_text.shape[0] == X_struct.shape[0] == len(idx) == len(games), \
       "Desajuste en número de filas entre matrices y games/idx"

# Alineación entre games e indice
assert (games["appid"].values == idx["appid"].values).all(), \
       "Desalineación entre games e idx (orden distinto)"

Formas:
X_text  : (2468, 20000)
X_struct: (2468, 12)
idx     : (2468, 2)
games   : (2468, 15)


In [11]:
# Comprobar juegos de user_games presentes en idx
missing_from_idx = set(user_games["appid"]) - set(idx["appid"])
print("Juegos en user_games no presentes en idx:", len(missing_from_idx))

assert len(missing_from_idx) == 0, "Hay juegos de user_games ausentes en idx"

Juegos en user_games no presentes en idx: 0


In [12]:
# Dispersión de las matrices
total_cells_tfidf = X_text.shape[0] * X_text.shape[1]
sparsity_tfidf = 1 - (X_text.nnz / total_cells_tfidf)

total_cells_struct = X_struct.shape[0] * X_struct.shape[1]
sparsity_struct = 1 - (X_struct.nnz / total_cells_struct)

print(f"Sparsity TF-IDF     : {sparsity_tfidf*100:.2f}%")
print(f"Sparsity estructural: {sparsity_struct*100:.2f}%")

Sparsity TF-IDF     : 99.03%
Sparsity estructural: 62.14%


In [13]:
# Verificar que no hay filas completamente vacías
zero_rows_tfidf = (X_text.getnnz(axis=1) == 0).sum()
zero_rows_struct = (X_struct.getnnz(axis=1) == 0).sum()

print("Juegos sin información TF-IDF:", zero_rows_tfidf)
print("Juegos sin información estructural:", zero_rows_struct)

assert zero_rows_tfidf == 0, "Existen juegos sin información TF-IDF"
assert zero_rows_struct == 0, "Existen juegos sin información estructural"

Juegos sin información TF-IDF: 0
Juegos sin información estructural: 0


# 🧊 Perfil de usuario

El objetivo del perfil de usuario es representar a cada usuario en el mismo espacio vectorial que los juegos. A partir de los juegos presentes en el conjunto de train se agregan sus representaciones de contenido para sintetizarlas en un único vector que capture sus preferencias. Este vector de perfil se compara con los vectores de todos los juegos del catálogo mediante la similitud del coseno para obtener el ranking de recomendaciones.

## Pesos a utilizar

La combinación entre las representaciones textual y estructural requiere determinar el peso relativo ($\alpha$) que debe asignarse a cada bloque. 

In [14]:
# Valores candidatos de alpha_text (peso del texto)
alphas = [0.9, 0.8, 0.7, 0.6, 0.5, 0.4, 0.3]

# Usaremos k=10 para seleccionar alpha
k_tuning = 10

In [15]:
# Función evaluar diferentes valores de alpha
results_alpha_list = []

for alpha_text in alphas:
    alpha_struct = 1.0 - alpha_text

    # Construir matriz combinada para este alpha
    X_combined_alpha = sparse.hstack([
        X_struct * alpha_struct,
        X_text   * alpha_text,
    ]).tocsr()
    X_combined_alpha = normalize(X_combined_alpha, norm="l2")

    print(f"Evaluando alpha_text={alpha_text:.1f}, alpha_struct={alpha_struct:.1f}...")

    # Evaluación LMO del modelo de contenido con este alpha
    metrics = evaluate_content_user_LMO_at_k(
        k=k_tuning,
        games=games,
        user_train=user_train,
        user_test=user_test,
        U_norm=None,
        user_to_pos=None,
        X_combined=X_combined_alpha,
        idx=idx,
        results_dir=results_dir,
        tag=f"alpha_{alpha_text:.1f}",
    )

    # Añadir alpha a las métricas
    metrics["alpha_text"] = float(alpha_text)
    metrics["alpha_struct"] = float(alpha_struct)
    results_alpha_list.append(metrics)

# DataFrame con resultados por alpha
results_alphas = pd.DataFrame(results_alpha_list)

# Guardar resultados del tuning supervisado
os.makedirs(results_dir, exist_ok=True)
results_alphas.to_csv(
    os.path.join(results_dir, f"alpha_tuning_content_LMO_k{k_tuning}.csv"),
    index=False,
)

# Mostrar tabla ordenada por NDCG, luego recall y precision
print(
    results_alphas.sort_values(
        ["ndcg@k", "recall@k", "precision@k"],
        ascending=False
    ).reset_index(drop=True).round(4)
)

# Seleccionar el mejor alpha según NDCG@k (y desempates)
best_row = results_alphas.sort_values(
    ["ndcg@k", "recall@k", "precision@k"],
    ascending=False
).iloc[0]

alpha_text = float(best_row["alpha_text"])
alpha_struct = float(best_row["alpha_struct"])

print(
    f"Alpha seleccionado (LMO, k={k_tuning}): "
    f"alpha_text={alpha_text:.2f}, alpha_struct={alpha_struct:.2f}"
)



Evaluando alpha_text=0.9, alpha_struct=0.1...
Evaluando alpha_text=0.8, alpha_struct=0.2...
Evaluando alpha_text=0.7, alpha_struct=0.3...
Evaluando alpha_text=0.6, alpha_struct=0.4...
Evaluando alpha_text=0.5, alpha_struct=0.5...
Evaluando alpha_text=0.4, alpha_struct=0.6...
Evaluando alpha_text=0.3, alpha_struct=0.7...
    k  precision@k  recall@k    f1@k  ndcg@k  hit_rate@k  item_coverage@k  \
0  10       0.0113    0.0376  0.0173  0.0570      0.0945           0.5814   
1  10       0.0117    0.0388  0.0179  0.0561      0.0970           0.5344   
2  10       0.0116    0.0386  0.0178  0.0556      0.0968           0.5182   
3  10       0.0096    0.0320  0.0148  0.0513      0.0831           0.6074   
4  10       0.0075    0.0250  0.0116  0.0397      0.0670           0.6207   
5  10       0.0061    0.0203  0.0093  0.0314      0.0554           0.5855   
6  10       0.0053    0.0177  0.0082  0.0262      0.0490           0.5389   

   num_users  alpha_text  alpha_struct  
0      47627        

## Matriz combinada

In [16]:
# Matriz combinada
X_combined = sparse.hstack([
    X_struct * alpha_struct,
    X_text   * alpha_text
]).tocsr()

X_combined = normalize(X_combined, norm="l2")

print("Forma X_combined:", X_combined.shape)

Forma X_combined: (2468, 20012)


In [17]:
# Mapas de índices
appid_to_row = {appid: i for i, appid in enumerate(idx["appid"].values)}

unique_users = user_train["user_id"].unique()
print("N usuarios únicos en TRAIN:", len(unique_users))

N usuarios únicos en TRAIN: 47627


# 📈 Evaluación del modelo

In [19]:
# Evaluar modelo de similitud pura con k = 10

evaluate_content_user_LMO_at_k(
    k=10,
    games=games,
    user_train=user_train,
    user_test=user_test,
    U_norm=None,
    user_to_pos=None,
    X_combined=X_combined,
    idx=idx,
    results_dir=results_dir,
    tag="20251130",
)


{'k': 10,
 'precision@k': 0.01127301740609318,
 'recall@k': 0.03757672468697727,
 'f1@k': 0.01734310370168182,
 'ndcg@k': 0.05701764142525676,
 'hit_rate@k': 0.09454721061582716,
 'item_coverage@k': 0.5814424635332253,
 'num_users': 47627}

In [20]:
evaluate_content_user_LMO_at_k(
    k=50,
    games=games,
    user_train=user_train,
    user_test=user_test,
    U_norm=None,
    user_to_pos=None,
    X_combined=X_combined,
    idx=idx,
    results_dir=results_dir,
    tag="20251130",
)

{'k': 50,
 'precision@k': 0.005352846074705524,
 'recall@k': 0.08921410124509205,
 'f1@k': 0.010099709574916084,
 'ndcg@k': 0.08243312736407592,
 'hit_rate@k': 0.21998026329602957,
 'item_coverage@k': 0.8885737439222042,
 'num_users': 47627}

# 💾 Guardar resultados

In [21]:
# Matriz combinada: Recomendación basada en contenido usuario-item
combined_path = os.path.join(models_content_dir, f"X_combined_alpha_{alpha_text:.1f}.npz")
sparse.save_npz(combined_path, X_combined)